# Figure 3 (Standalone) - Falsifying Simple Explanations of Plasticity Loss

**Self-contained version for Google Colab** — every function this experiment needs (environments, models, replay buffer, DQN loop, plasticity probe, statistics) is defined inline below; there are no imports from the repository's `src/` package. Upload this single file to Colab and run it.

**Colab setup**: `Runtime → Change runtime type → GPU` (a T4 is plenty). All required libraries (torch, torchvision, numpy, pandas, matplotlib, tqdm) are preinstalled on Colab.

**What it reproduces**: Section 5.1 of Lyle et al., *Understanding Plasticity in Neural Networks*. Train a DQN agent on a classification MDP (`easy`/`hard`/`sparse` over MNIST or CIFAR-10 observations); every `probe_every` steps, pause to (a) run the random-target plasticity probe and (b) recompute four candidate statistics (weight norm, weight rank, dead units, feature rank). If a statistic really explained plasticity loss, its correlation would be consistently strong and same-signed across every controlled comparison — the paper's finding is that it isn't.

**Chunked execution**: every config is an independent run, so you can split the paper sweep across several Colab/Kaggle sessions (by architecture, seed, dataset — anything) and merge the CSVs afterwards. See the "Chunked Paper-Scale Run" and "Merging Chunks" sections at the bottom.

## Setup

Paths are relative to the working directory (`/content` on Colab). Datasets download automatically on first use; results land in `outputs/tables` and `outputs/figures` — download them via Colab's file browser (left sidebar) when the run finishes.

In [1]:
import copy
import random
import time
from dataclasses import dataclass, replace
from pathlib import Path
from typing import Callable, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from tqdm.auto import tqdm

ROOT = Path.cwd()
DATA_ROOT = ROOT / "data"
OUTPUT_ROOT = ROOT / "outputs"
FIGURES_DIR = OUTPUT_ROOT / "figures"
TABLES_DIR = OUTPUT_ROOT / "tables"
for directory in (DATA_ROOT, OUTPUT_ROOT, FIGURES_DIR, TABLES_DIR):
    directory.mkdir(parents=True, exist_ok=True)


def _default_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():  # Apple-silicon GPU
        return torch.device("mps")
    return torch.device("cpu")


DEVICE = _default_device()
print(f"Using device: {DEVICE}")
if DEVICE.type == "cuda":
    print(torch.cuda.get_device_name(0))

Using device: mps


/Users/mahdigheidi/Documents/Univ/Masters-Study-Project/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Environments: the Classification MDPs (paper Section 3.2)

MNIST/CIFAR-10 classification recast as a ten-state, ten-action block MDP. A latent state selects a class-conditional image observation; the agent predicts an action in `{0..9}`; the reward rule defines the variant: **easy** (true labels, reward = correct prediction), **hard** (random labels), **sparse** (reward only at state 9 → action 9, with sequential state progression).

In [2]:
@dataclass(frozen=True)
class ClassificationMDPSpec:
    name: str
    num_states: int = 10
    num_actions: int = 10


class ClassificationMDP:
    def __init__(self, dataset, spec, labels=None, seed=None):
        self.dataset = dataset
        self.spec = spec
        self.rng = random.Random(seed)
        self.labels = labels if labels is not None else self._true_labels(dataset)
        self.class_indices = self._build_class_indices(self.labels)
        self.state = self.rng.randrange(self.spec.num_states)

    @staticmethod
    def _true_labels(dataset):
        targets = getattr(dataset, "targets", None)
        if targets is None:
            return [int(dataset[idx][1]) for idx in range(len(dataset))]
        if isinstance(targets, torch.Tensor):
            return [int(value) for value in targets.tolist()]
        return [int(value) for value in targets]

    def _build_class_indices(self, labels):
        class_indices = {state: [] for state in range(self.spec.num_states)}
        for idx, raw_label in enumerate(labels):
            label = int(raw_label)
            if 0 <= label < self.spec.num_states:
                class_indices[label].append(idx)
        empty = [state for state, indices in class_indices.items() if not indices]
        if empty:
            raise ValueError(f"Every MDP state must have at least one image. Missing states: {empty}")
        return class_indices

    def reset(self, state=None):
        self.state = self.rng.randrange(self.spec.num_states) if state is None else int(state)
        return self.sample_observation(self.state)

    def sample_observation(self, state=None):
        state = self.state if state is None else int(state)
        idx = self.rng.choice(self.class_indices[state])
        image, _ = self.dataset[idx]
        return image

    def transition(self, action):
        raise NotImplementedError

    def step(self, action):
        reward, next_state = self.transition(int(action))
        self.state = int(next_state)
        return self.sample_observation(self.state), float(reward), self.state


def make_random_labels(dataset, num_states=10, seed=None):
    rng = random.Random(seed)
    return [rng.randrange(num_states) for _ in range(len(dataset))]


class EasyMDP(ClassificationMDP):
    def __init__(self, dataset, seed=None):
        super().__init__(dataset, ClassificationMDPSpec(name="easy"), seed=seed)

    def transition(self, action):
        reward = float(int(action) == int(self.state))
        self.state = self.rng.randrange(self.spec.num_states)
        return reward, self.state


class HardMDP(ClassificationMDP):
    def __init__(self, dataset, seed=None):
        labels = make_random_labels(dataset, seed=seed)
        super().__init__(dataset, ClassificationMDPSpec(name="hard"), labels=labels, seed=seed)

    def transition(self, action):
        reward = float(int(action) == int(self.state))
        self.state = self.rng.randrange(self.spec.num_states)
        return reward, self.state


class SparseMDP(ClassificationMDP):
    def __init__(self, dataset, seed=None):
        super().__init__(dataset, ClassificationMDPSpec(name="sparse"), seed=seed)

    def transition(self, action):
        action_matches_state = int(action) == int(self.state)
        reward = float(action_matches_state and int(self.state) == 9)
        if action_matches_state:
            next_state = (int(self.state) + 1) % self.spec.num_states
        else:
            next_state = self.rng.randrange(self.spec.num_states)
        self.state = next_state
        return reward, next_state

## Models (paper Appendix A.2)

Two-hidden-layer ReLU MLP (width configurable) and a two-conv/two-fc CNN. Both expose `forward_features` (the penultimate representation) for the feature-rank metric, and use module `nn.ReLU`s so the dead-unit metric can hook them.

In [3]:
def _prod(values):
    out = 1
    for v in values:
        out *= int(v)
    return out


def _maybe_spectral_norm(module, enabled):
    return nn.utils.spectral_norm(module) if enabled else module


class MLP(nn.Module):
    def __init__(self, input_shape=(1, 28, 28), num_actions=10, hidden_dim=512,
                 use_layernorm=False, spectral_norm=False):
        super().__init__()
        self.input_dim = _prod(input_shape)
        self.hidden_dim = int(hidden_dim)
        self.num_actions = int(num_actions)
        self.feature_dim = self.hidden_dim

        self.fc1 = _maybe_spectral_norm(nn.Linear(self.input_dim, self.hidden_dim), spectral_norm)
        self.fc2 = _maybe_spectral_norm(nn.Linear(self.hidden_dim, self.hidden_dim), spectral_norm)
        self.output = nn.Linear(self.hidden_dim, self.num_actions)

        self.ln1 = nn.LayerNorm(self.hidden_dim) if use_layernorm else nn.Identity()
        self.ln2 = nn.LayerNorm(self.hidden_dim) if use_layernorm else nn.Identity()
        self.relu1 = nn.ReLU()
        self.relu2 = nn.ReLU()

    def forward_features(self, x):
        x = x.view(x.size(0), -1)
        x = self.relu1(self.ln1(self.fc1(x)))
        x = self.relu2(self.ln2(self.fc2(x)))
        return x

    def forward(self, x):
        return self.output(self.forward_features(x))


class CNN(nn.Module):
    def __init__(self, input_shape=(1, 28, 28), num_actions=10, conv_channels=64,
                 fc_dim=256, use_layernorm=False, spectral_norm=False):
        super().__init__()
        input_channels = int(input_shape[0])
        self.input_shape = tuple(int(v) for v in input_shape)
        self.num_actions = int(num_actions)
        self.conv_channels = int(conv_channels)
        self.fc_dim = int(fc_dim)
        self.feature_dim = self.fc_dim

        self.conv1 = _maybe_spectral_norm(nn.Conv2d(input_channels, self.conv_channels, kernel_size=5), spectral_norm)
        self.conv2 = _maybe_spectral_norm(nn.Conv2d(self.conv_channels, self.conv_channels, kernel_size=3), spectral_norm)
        self.relu1 = nn.ReLU()
        self.relu2 = nn.ReLU()
        self.flatten = nn.Flatten()

        with torch.no_grad():
            dummy = torch.zeros(1, *self.input_shape)
            conv1_shape = self.conv1(dummy).shape[1:]
            conv2_shape = self.conv2(self.relu1(self.conv1(dummy))).shape[1:]
            conv_dim = int(torch.numel(torch.zeros(conv2_shape)))

        self.ln1 = nn.LayerNorm(conv1_shape) if use_layernorm else nn.Identity()
        self.ln2 = nn.LayerNorm(conv2_shape) if use_layernorm else nn.Identity()

        self.fc1 = _maybe_spectral_norm(nn.Linear(conv_dim, self.fc_dim), spectral_norm)
        self.fc2 = _maybe_spectral_norm(nn.Linear(self.fc_dim, self.fc_dim), spectral_norm)
        self.output = nn.Linear(self.fc_dim, self.num_actions)

        self.ln3 = nn.LayerNorm(self.fc_dim) if use_layernorm else nn.Identity()
        self.ln4 = nn.LayerNorm(self.fc_dim) if use_layernorm else nn.Identity()
        self.relu3 = nn.ReLU()
        self.relu4 = nn.ReLU()

    def forward_features(self, x):
        x = self.relu1(self.ln1(self.conv1(x)))
        x = self.relu2(self.ln2(self.conv2(x)))
        x = self.flatten(x)
        x = self.relu3(self.ln3(self.fc1(x)))
        x = self.relu4(self.ln4(self.fc2(x)))
        return x

    def forward(self, x):
        return self.output(self.forward_features(x))

## Replay Buffer

In [4]:
class ReplayBuffer:
    def __init__(self, capacity):
        from collections import deque
        self.capacity = capacity
        self.buffer = deque(maxlen=capacity)
        self.device = DEVICE

    def push(self, image, action, reward, next_image):
        self.buffer.append((image, action, reward, next_image))

    def sample(self, batch_size, device=None):
        device = self.device if device is None else device
        batch = random.sample(self.buffer, batch_size)
        images, actions, rewards, next_images = zip(*batch)
        return (
            torch.tensor(np.array(images), dtype=torch.float32, device=device),
            torch.tensor(actions, dtype=torch.long, device=device),
            torch.tensor(rewards, dtype=torch.float32, device=device),
            torch.tensor(np.array(next_images), dtype=torch.float32, device=device),
        )

    def sample_states(self, batch_size, device=None):
        device = self.device if device is None else device
        batch = random.sample(self.buffer, min(batch_size, len(self.buffer)))
        images = [item[0] for item in batch]
        return torch.tensor(np.array(images), dtype=torch.float32, device=device)

    def __len__(self):
        return len(self.buffer)

## DQN Training Loop (shared machinery)

In [5]:
@dataclass
class ClassificationDQNConfig:
    seed: int = 0
    data_root: str = "./data"
    download: bool = True
    observation_space: str = "mnist"
    environment: str = "easy"
    architecture: str = "mlp"
    hidden_dim: int = 512
    cnn_channels: int = 64
    cnn_fc_dim: int = 256
    gamma: float = 0.99
    lr: float = 1e-3
    optimizer: str = "adam"
    weight_decay: float = 0.0
    batch_size: int = 512
    replay_capacity: int = 50_000
    warmup_steps: int = 2_000
    train_steps: int = 20_000
    target_update_period: int = 1_000
    epsilon_start: float = 1.0
    epsilon_final: float = 0.1
    epsilon_decay: int = 10_000
    use_layernorm: bool = False
    spectral_norm: bool = False


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_dataset(config):
    if config.observation_space == "mnist":
        dataset = datasets.MNIST(root=str(config.data_root), train=True,
                                 download=config.download, transform=transforms.ToTensor())
        input_shape = (1, 28, 28)
    elif config.observation_space == "cifar10":
        dataset = datasets.CIFAR10(root=str(config.data_root), train=True,
                                   download=config.download, transform=transforms.ToTensor())
        input_shape = (3, 32, 32)
    else:
        raise ValueError(f"Unknown observation space: {config.observation_space}")
    return dataset, input_shape


def build_environment(config, dataset):
    if config.environment == "easy":
        return EasyMDP(dataset, seed=config.seed)
    if config.environment == "hard":
        return HardMDP(dataset, seed=config.seed)
    if config.environment == "sparse":
        return SparseMDP(dataset, seed=config.seed)
    raise ValueError(f"Unknown environment: {config.environment}")


def build_model_factory(config, input_shape):
    def factory():
        if config.architecture == "mlp":
            return MLP(input_shape=input_shape, num_actions=10, hidden_dim=config.hidden_dim,
                       use_layernorm=config.use_layernorm, spectral_norm=config.spectral_norm)
        if config.architecture == "cnn":
            return CNN(input_shape=input_shape, num_actions=10, conv_channels=config.cnn_channels,
                       fc_dim=config.cnn_fc_dim, use_layernorm=config.use_layernorm,
                       spectral_norm=config.spectral_norm)
        raise ValueError(f"Unknown architecture: {config.architecture}")
    return factory


def build_optimizer(config, model):
    if config.optimizer == "adam":
        return torch.optim.Adam(model.parameters(), lr=config.lr, weight_decay=config.weight_decay)
    if config.optimizer == "sgd":
        return torch.optim.SGD(model.parameters(), lr=config.lr, weight_decay=config.weight_decay)
    raise ValueError(f"Unknown optimizer: {config.optimizer}")


def optimizer_factory_from_config(config):
    def factory(model):
        return build_optimizer(config, model)
    return factory


def epsilon_at_step(config, step):
    return config.epsilon_final + (config.epsilon_start - config.epsilon_final) * np.exp(
        -float(step) / float(config.epsilon_decay)
    )


@torch.no_grad()
def select_action(model, observation, epsilon, device=None):
    device = DEVICE if device is None else device
    if random.random() < epsilon:
        return random.randrange(10)
    q_values = model(observation.unsqueeze(0).to(device))
    return int(q_values.argmax(dim=1).item())


def collect_transition(env, model, replay, epsilon, device=None):
    observation = env.sample_observation()
    action = select_action(model, observation, epsilon, device=device)
    next_observation, reward, _ = env.step(action)
    replay.push(observation, action, reward, next_observation)


def dqn_loss(model, target_model, batch, gamma):
    states, actions, rewards, next_states = batch
    q_sa = model(states).gather(1, actions.unsqueeze(1)).squeeze(1)
    with torch.no_grad():
        next_q = target_model(next_states).max(dim=1).values
        target = rewards + gamma * next_q
    return F.mse_loss(q_sa, target)


def train_dqn_step(model, target_model, optimizer, replay, config, device=None):
    device = DEVICE if device is None else device
    batch = replay.sample(config.batch_size, device=device)
    loss = dqn_loss(model, target_model, batch, config.gamma)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    return float(loss.detach().cpu().item())

## Plasticity Probe (paper Sections 2.2 / 3.1)

For a probe batch `X`, sample a fresh network `f_omega` and build random regression targets `a + sin(1e5 * f_omega(X))` (`a` = the current network's mean prediction). A *clone* of the checkpoint is optimized on this target for a fixed budget; the final loss, averaged over `num_tasks` random targets, is the probe loss. Plasticity loss = probe loss now minus probe loss at initialization.

In [6]:
@dataclass
class PlasticityProbeConfig:
    steps: int = 500
    num_tasks: int = 10
    batch_size: Optional[int] = None
    target_scale: float = 1e5
    log_every: Optional[int] = None


@dataclass
class ProbeTaskResult:
    initial_loss: float
    final_loss: float
    learning_curve: List[Dict[str, float]]


def _device_of(model):
    return next(model.parameters()).device


def _sample_indices(inputs, batch_size):
    if batch_size is None or batch_size >= inputs.size(0):
        return None
    return torch.randint(0, inputs.size(0), (batch_size,), device=inputs.device)


@torch.no_grad()
def make_random_function_targets(model, random_model, inputs, target_scale=1e5):
    model.eval()
    random_model.eval()
    inputs = inputs.to(_device_of(model))
    random_model = random_model.to(inputs.device)
    offset = model(inputs).mean(dim=0, keepdim=True)
    random_outputs = random_model(inputs)
    return (offset + torch.sin(target_scale * random_outputs)).detach()


def train_probe_task(model, inputs, targets, optimizer_factory, config, progress_desc=None):
    probe_model = copy.deepcopy(model).to(_device_of(model))
    inputs = inputs.to(_device_of(probe_model))
    targets = targets.to(_device_of(probe_model))
    optimizer = optimizer_factory(probe_model)

    with torch.no_grad():
        initial_loss = float(F.mse_loss(probe_model(inputs), targets).cpu().item())

    probe_model.train()
    step_iter = tqdm(range(1, config.steps + 1), desc=progress_desc, unit="step",
                     leave=False, disable=progress_desc is None)
    for step in step_iter:
        indices = _sample_indices(inputs, config.batch_size)
        if indices is None:
            batch_inputs, batch_targets = inputs, targets
        else:
            batch_inputs = inputs.index_select(0, indices)
            batch_targets = targets.index_select(0, indices)

        loss = F.mse_loss(probe_model(batch_inputs), batch_targets)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        if progress_desc is not None and step % 50 == 0:
            step_iter.set_postfix(loss=f"{float(loss.detach().cpu()):.4f}")

    with torch.no_grad():
        final_loss = float(F.mse_loss(probe_model(inputs), targets).cpu().item())

    return ProbeTaskResult(initial_loss=initial_loss, final_loss=final_loss, learning_curve=[])


def run_random_probe_task(model, random_model_factory, inputs, optimizer_factory, config, progress_desc=None):
    random_model = random_model_factory().to(_device_of(model))
    plasticity_targets = make_random_function_targets(model, random_model, inputs, target_scale=config.target_scale)
    return train_probe_task(model, inputs, plasticity_targets, optimizer_factory, config, progress_desc=progress_desc)


def estimate_probe_loss(model, random_model_factory, inputs, optimizer_factory, config, desc=None):
    return [
        run_random_probe_task(
            model, random_model_factory, inputs, optimizer_factory, config,
            progress_desc=None if desc is None else f"{desc} [task {task + 1}/{config.num_tasks}]",
        )
        for task in range(config.num_tasks)
    ]

## The Four Candidate Statistics

Weight norm (global L2), weight rank (mean numerical rank of linear/conv kernels), dead units (ReLU units never positive on a probe batch), and feature rank (numerical rank of the centered penultimate representation). SVDs run on CPU deliberately — `svdvals` isn't implemented on all accelerators (e.g. MPS) and the matrices are small.

In [7]:
@torch.no_grad()
def compute_weight_norm(model, include_bias=True):
    total_norm_sq = 0.0
    for name, parameter in model.named_parameters():
        if not include_bias and name.endswith("bias"):
            continue
        if parameter.requires_grad:
            total_norm_sq += float(parameter.detach().pow(2).sum().cpu().item())
    return float(total_norm_sq ** 0.5)


@torch.no_grad()
def matrix_rank(weight_matrix, threshold=1e-5):
    matrix = weight_matrix.detach().cpu()
    if matrix.dim() > 2:
        matrix = matrix.reshape(matrix.size(0), -1)
    singular_values = torch.linalg.svdvals(matrix)
    return int((singular_values > threshold).sum().item())


@torch.no_grad()
def compute_weight_rank(model, threshold=1e-5):
    ranks = [
        float(matrix_rank(module.weight, threshold=threshold))
        for module in model.modules()
        if isinstance(module, (nn.Linear, nn.Conv2d))
    ]
    return float(np.mean(ranks)) if ranks else 0.0


def _unit_alive(activation):
    if activation.dim() == 2:
        return (activation > 0).any(dim=0)
    return (activation > 0).any(dim=(0, *range(2, activation.dim())))


@torch.no_grad()
def compute_dead_units(model, data, device=None):
    if device is None:
        device = _device_of(model)
    model.eval()
    alive_by_layer = {}
    latest_activations = {}
    hooks = []

    def hook_fn(name):
        def hook(_, __, output):
            latest_activations[name] = output.detach()
        return hook

    for name, module in model.named_modules():
        if isinstance(module, nn.ReLU):
            hooks.append(module.register_forward_hook(hook_fn(name)))

    try:
        batches = [data] if isinstance(data, torch.Tensor) else data
        for batch in batches:
            latest_activations.clear()
            model(batch.to(device))
            for name, activation in latest_activations.items():
                alive = _unit_alive(activation).detach().cpu()
                if name not in alive_by_layer:
                    alive_by_layer[name] = alive.clone()
                else:
                    alive_by_layer[name] |= alive
    finally:
        for hook_handle in hooks:
            hook_handle.remove()

    return float(sum((~alive).sum().item() for alive in alive_by_layer.values()))


@torch.no_grad()
def compute_model_feature_rank(model, data, threshold=0.01, device=None):
    if device is None:
        device = _device_of(model)
    model.eval()
    batches = [data] if isinstance(data, torch.Tensor) else data
    features = torch.cat(
        [model.forward_features(batch.to(device)).detach().cpu() for batch in batches], dim=0
    )
    if features.dim() > 2:
        features = features.flatten(start_dim=1)
    features = features - features.mean(dim=0, keepdim=True)
    singular_values = torch.linalg.svdvals(features)
    return int((singular_values > threshold).sum().item())

## Figure 3 Experiment: Config, Panels, and the Train + Probe Loop

In [8]:
@dataclass
class Figure3RunConfig(ClassificationDQNConfig):
    probe_every: int = 5_000
    probe_steps: int = 2_000
    num_probe_tasks: int = 10
    probe_batch_size: int = 512
    metric_batch_size: int = 512


FIGURE3_SCHEMA = [
    "run_id", "seed", "observation_space", "environment", "architecture", "optimizer",
    "step", "probe_loss", "initial_probe_loss", "plasticity_loss",
    "weight_norm", "weight_rank", "dead_units", "feature_rank",
]


@dataclass(frozen=True)
class PanelSpec:
    panel: str
    title: str
    x_column: str
    x_label: str
    condition_column: str
    conditions: Tuple[str, ...]


PANEL_SPECS = (
    PanelSpec("weight_norm", "Varying observation space", "weight_norm", "Weight norm",
              "observation_space", ("cifar10", "mnist")),
    PanelSpec("weight_rank", "Varying observation space", "weight_rank", "Weight rank",
              "observation_space", ("cifar10", "mnist")),
    PanelSpec("dead_units", "Varying architecture", "dead_units", "Dead units",
              "architecture", ("cnn", "mlp")),
    PanelSpec("feature_rank", "Varying reward function", "feature_rank", "Feature rank",
              "environment", ("easy", "hard", "sparse")),
)

COLORS = {
    "cifar10": "#23c4b6", "mnist": "#6d3df2",
    "cnn": "#23c4b6", "mlp": "#6d3df2",
    "easy": "#23c4b6", "hard": "#6d3df2", "sparse": "#e83f76",
}


def _has_regression_support(x, y):
    return len(x) >= 2 and float(np.std(x)) > 1e-12 and float(np.std(y)) > 1e-12

In [9]:

@torch.no_grad()
def feature_singular_values(features: torch.Tensor, center: bool = True) -> torch.Tensor:
    """Singular values of the feature matrix, descending.

    Every rank definition in this module is a different way of reading this one
    spectrum, so it is exposed to let callers inspect it directly.
    """
    if features.dim() > 2:
        features = features.flatten(start_dim=1)
    if center:
        features = features - features.mean(dim=0, keepdim=True)
    # svdvals is not implemented for MPS tensors, so measure on the CPU copy.
    return torch.linalg.svdvals(features.detach().cpu())


@torch.no_grad()
def compute_feature_rank(features: torch.Tensor, threshold: float = 1e-5) -> int:
    singular_values = feature_singular_values(features)
    return int((singular_values > threshold).sum().item())


@torch.no_grad()
def compute_feature_srank(
    model,
    data,
    delta: float = 0.01,
    center: bool = True,
) -> int:
    """Effective rank of Kumar et al. (2020), cited by Lyle et al. for Figure 3.

    Returns the smallest ``k`` whose top-``k`` singular values carry at least
    ``1 - delta`` of the total singular-value mass.  Where ``compute_feature_rank``
    applies an absolute threshold and therefore counts every direction that is
    merely non-zero, this measures where the representation's energy is actually
    concentrated.

    Kumar et al. define srank on the raw feature matrix; ``center`` stays
    configurable because ReLU features are non-negative and so carry a large mean
    component that would otherwise dominate the spectrum.
    """
    device = _device_of(model)
    model.eval()
    batches = [data] if isinstance(data, torch.Tensor) else data
    features = torch.cat(
        [model.forward_features(batch.to(device)).detach().cpu() for batch in batches], dim=0
    )

    if not 0.0 <= delta < 1.0:
        raise ValueError(f"delta must lie in [0, 1), got {delta}.")

    singular_values = feature_singular_values(features, center=center)
    total = float(singular_values.sum().item())
    if total <= 0.0:
        return 0

    cumulative = torch.cumsum(singular_values, dim=0) / total
    below_threshold = int((cumulative < (1.0 - delta)).sum().item())
    return min(below_threshold + 1, int(singular_values.numel()))

In [10]:
def run_id_for(config):
    return (f"{config.observation_space}_{config.environment}_"
            f"{config.architecture}_{config.optimizer}_seed{config.seed}")


def _mean_probe_loss(results):
    return float(np.mean([result.final_loss for result in results]))


def compute_probe_loss(model, model_factory, optimizer_factory, replay, config, desc=None):
    states = replay.sample_states(config.probe_batch_size, device=DEVICE)
    probe_config = PlasticityProbeConfig(
        steps=config.probe_steps, num_tasks=config.num_probe_tasks, batch_size=config.probe_batch_size,
    )
    estimated_probe_losses = estimate_probe_loss(model, model_factory, states, optimizer_factory, probe_config, desc=desc)
    return _mean_probe_loss(estimated_probe_losses)


def metric_row(run_id, config, step, model, model_factory, optimizer_factory, replay,
               initial_probe_loss, probe_loss_override=None):
    metric_states = replay.sample_states(config.metric_batch_size, device=DEVICE)
    probe_loss = (
        compute_probe_loss(model, model_factory, optimizer_factory, replay, config,
                           desc=f"probe @ step {step}")
        if probe_loss_override is None
        else probe_loss_override
    )
    row = {
        "run_id": run_id,
        "seed": config.seed,
        "observation_space": config.observation_space,
        "environment": config.environment,
        "architecture": config.architecture,
        "optimizer": config.optimizer,
        "step": step,
        "probe_loss": probe_loss,
        "initial_probe_loss": initial_probe_loss,
        "plasticity_loss": probe_loss - initial_probe_loss,
        "weight_norm": compute_weight_norm(model),
        "weight_rank": compute_weight_rank(model),
        "dead_units": compute_dead_units(model, metric_states),
        "feature_rank": compute_feature_srank(model, metric_states),
    }
    print(f"    step {step:>6}: probe_loss={row['probe_loss']:.4f} "
          f"plasticity_loss={row['plasticity_loss']:+.4f} "
          f"weight_norm={row['weight_norm']:.2f} weight_rank={row['weight_rank']:.1f} "
          f"dead_units={row['dead_units']:.0f} feature_rank={row['feature_rank']}")
    return row


def run_training_config(config):
    run_id = run_id_for(config)
    set_seed(config.seed)
    dataset, input_shape = load_dataset(config)
    env = build_environment(config, dataset)
    model_factory = build_model_factory(config, input_shape)
    model = model_factory().to(DEVICE)
    target_model = copy.deepcopy(model).to(DEVICE)
    optimizer = build_optimizer(config, model)
    optimizer_factory = optimizer_factory_from_config(config)
    replay = ReplayBuffer(config.replay_capacity)

    for _ in tqdm(range(config.warmup_steps), desc="warmup", unit="step", leave=False):
        collect_transition(env, model, replay, epsilon=1.0, device=DEVICE)

    initial_probe_loss = compute_probe_loss(
        model, model_factory, optimizer_factory, replay, config, desc="initial probe"
    )

    rows = [metric_row(run_id, config, 0, model, model_factory, optimizer_factory, replay,
                       initial_probe_loss, probe_loss_override=initial_probe_loss)]

    last_loss = float("nan")
    train_bar = tqdm(range(1, config.train_steps + 1), desc=f"train {run_id}", unit="step")
    for step in train_bar:
        epsilon = epsilon_at_step(config, step)
        collect_transition(env, model, replay, epsilon=epsilon, device=DEVICE)
        last_loss = train_dqn_step(model, target_model, optimizer, replay, config, device=DEVICE)

        if step % 100 == 0:
            train_bar.set_postfix(loss=f"{last_loss:.4f}", eps=f"{epsilon:.2f}")

        if step % config.target_update_period == 0:
            target_model.load_state_dict(model.state_dict())

        if step % config.probe_every == 0:
            rows.append(metric_row(run_id, config, step, model, model_factory,
                                   optimizer_factory, replay, initial_probe_loss))
    train_bar.close()

    return pd.DataFrame(rows, columns=FIGURE3_SCHEMA)

## Config Factories and the Sweep Runner

In [11]:
def make_smoke_configs(data_root="./data", download=True):
    base = Figure3RunConfig(
        data_root=data_root, download=download,
        hidden_dim=128, batch_size=128, replay_capacity=5_000, warmup_steps=512,
        train_steps=1_000, target_update_period=250, probe_every=500, probe_steps=25,
        num_probe_tasks=2, probe_batch_size=128, metric_batch_size=128,
    )
    return [
        replace(base, seed=0, observation_space="mnist", environment="easy", architecture="mlp"),
        replace(base, seed=1, observation_space="mnist", environment="hard", architecture="mlp"),
        replace(base, seed=2, observation_space="mnist", environment="sparse", architecture="mlp"),
        replace(base, seed=3, observation_space="mnist", environment="easy", architecture="cnn"),
    ]


def make_paper_configs(data_root="./data", download=True):
    configs = []
    seeds = [random.randint(0, 10000) for _ in range(10)]  # Generate random seeds for reproducibility
    for seed in seeds:
        for observation_space in ["mnist", "cifar10"]:
            for environment in ["easy", "hard", "sparse"]:
                for architecture in ["cnn", "mlp"]:
                    configs.append(Figure3RunConfig(
                        seed=seed, data_root=data_root, download=download,
                        observation_space=observation_space, environment=environment,
                        architecture=architecture, hidden_dim=512, train_steps=50_000,
                        target_update_period=1_000, probe_every=5_000, probe_steps=2_000,
                        num_probe_tasks=10, probe_batch_size=512, metric_batch_size=512,
                    ))
    return configs


def run_sweep(configs, save_path=None):
    frames = []
    sweep_start = time.perf_counter()
    for idx, config in enumerate(configs, start=1):
        print(f"[{idx}/{len(configs)}] {run_id_for(config)}")
        config_start = time.perf_counter()
        frames.append(run_training_config(config))
        config_min = (time.perf_counter() - config_start) / 60
        elapsed_min = (time.perf_counter() - sweep_start) / 60
        remaining_min = elapsed_min / idx * (len(configs) - idx)
        print(f"[{idx}/{len(configs)}] done in {config_min:.1f} min | "
              f"elapsed {elapsed_min:.1f} min | est. remaining {remaining_min:.0f} min")
    df = pd.concat(frames, ignore_index=True)
    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(save_path, index=False)
        print(f"Saved summary rows to {save_path}")
    return df

## Smoke Run (fast sanity check)

Four tiny configs, a couple of minutes on a GPU — verifies the whole pipeline before you commit a session to the paper-scale sweep. Safe to skip once you trust the setup.

In [12]:
smoke_df = run_sweep(make_smoke_configs(data_root=str(DATA_ROOT)),
                     save_path=TABLES_DIR / "figure3_smoke.csv")
smoke_df.head()

[1/4] mnist_easy_mlp_adam_seed0


    step      0: probe_loss=0.2370 plasticity_loss=+0.0000 weight_norm=9.45 weight_rank=88.7 dead_units=9 feature_rank=96


train mnist_easy_mlp_adam_seed0:  53%|█████▎    | 533/1000 [00:02<00:03, 155.36step/s, eps=0.96, loss=0.0096]

    step    500: probe_loss=0.2821 plasticity_loss=+0.0451 weight_norm=13.35 weight_rank=88.7 dead_units=33 feature_rank=96


train mnist_easy_mlp_adam_seed0: 100%|██████████| 1000/1000 [00:04<00:00, 205.31step/s, eps=0.91, loss=0.0185]


...

train mnist_easy_cnn_adam_seed3:  50%|█████     | 501/1000 [00:29<03:09,  2.64step/s, eps=0.96, loss=0.0049]

    step    500: probe_loss=0.1171 plasticity_loss=+0.0372 weight_norm=32.25 weight_rank=122.2 dead_units=284 feature_rank=106


train mnist_easy_cnn_adam_seed3: 100%|██████████| 1000/1000 [00:52<00:00, 19.00step/s, eps=0.91, loss=0.0023]

    step   1000: probe_loss=0.2187 plasticity_loss=+0.1388 weight_norm=45.94 weight_rank=122.2 dead_units=309 feature_rank=98
[4/4] done in 0.9 min | elapsed 1.2 min | est. remaining 0 min
Saved summary rows to /Users/mahdigheidi/Documents/Univ/Masters-Study-Project/notebooks/outputs/tables/figure3_smoke.csv


,run_id,seed,observation_space,environment,architecture,optimizer,step,probe_loss,initial_probe_loss,plasticity_loss,weight_norm,weight_rank,dead_units,feature_rank
0,mnist_easy_mlp_adam_seed0,0,mnist,easy,mlp,adam,0,0.237004,0.237004,0.000000,9.452216,88.666667,9.0,96
1,mnist_easy_mlp_adam_seed0,0,mnist,easy,mlp,adam,500,0.282076,0.237004,0.045071,13.348791,88.666667,33.0,96
2,mnist_easy_mlp_adam_seed0,0,mnist,easy,mlp,adam,1000,0.285711,0.237004,0.048707,16.520708,88.666667,35.0,97
3,mnist_hard_mlp_adam_seed1,1,mnist,hard,mlp,adam,0,0.231781,0.231781,0.000000,9.449522,88.666667,6.0,100
4,mnist_hard_mlp_adam_seed1,1,mnist,hard,mlp,adam,500,0.288769,0.231781,0.056988,13.573909,88.666667,45.0,97


## Chunked Paper-Scale Run

Every config is fully independent, so split the 48-config paper sweep across sessions along **any** axis — architecture, seed, dataset, environment — and merge the CSVs afterwards. Edit the filter below for this session's slice and **give each chunk a distinct `save_path`**. Rough per-config cost on a Colab T4: MLP ≈ 1–3 min, CNN ≈ 5–25 min.

Suggested split: put CNN configs on GPU sessions (they're 50-100x slower than MLPs on CPU) and keep MLP chunks wherever is convenient.

In [13]:
all_configs = make_paper_configs(data_root=str(DATA_ROOT))
print(f"total paper configs: {len(all_configs)}")

chunk = [c for c in all_configs if c.architecture == "mlp" and c.observation_space == "mnist" and c.environment == "easy"]
chunk_name = "mlp_mnist_easy"  # EDIT: used in the output filename

print(f"this session runs {len(chunk)} configs: {[run_id_for(c) for c in chunk]}")

# Uncomment to launch:
chunk_df = run_sweep(chunk, save_path=TABLES_DIR / f"figure3_chunk_{chunk_name}.csv")

total paper configs: 120
this session runs 10 configs: ['mnist_easy_mlp_adam_seed8688', 'mnist_easy_mlp_adam_seed9781', 'mnist_easy_mlp_adam_seed4665', 'mnist_easy_mlp_adam_seed3187', 'mnist_easy_mlp_adam_seed6738', 'mnist_easy_mlp_adam_seed8335', 'mnist_easy_mlp_adam_seed8910', 'mnist_easy_mlp_adam_seed5635', 'mnist_easy_mlp_adam_seed9447', 'mnist_easy_mlp_adam_seed1936']
[1/10] mnist_easy_mlp_adam_seed8688


    step      0: probe_loss=0.0002 plasticity_loss=+0.0000 weight_norm=18.59 weight_rank=344.7 dead_units=3 feature_rank=420


train mnist_easy_mlp_adam_seed8688:  10%|█         | 5006/50000 [01:43<17:57:08,  1.44s/step, eps=0.65, loss=0.0049]

    step   5000: probe_loss=0.0001 plasticity_loss=-0.0001 weight_norm=46.04 weight_rank=344.7 dead_units=416 feature_rank=334


train mnist_easy_mlp_adam_seed8688:  20%|██        | 10006/50000 [03:32<16:09:53,  1.46s/step, eps=0.43, loss=0.0124]

    step  10000: probe_loss=0.0000 plasticity_loss=-0.0002 weight_norm=60.90 weight_rank=344.7 dead_units=418 feature_rank=347


train mnist_easy_mlp_adam_seed8688:  30%|███       | 15005/50000 [05:21<16:41:42,  1.72s/step, eps=0.30, loss=0.0324]

    step  15000: probe_loss=0.0000 plasticity_loss=-0.0002 weight_norm=72.29 weight_rank=344.7 dead_units=412 feature_rank=351


train mnist_easy_mlp_adam_seed8688:  33%|███▎      | 16569/50000 [05:39<11:23, 48.88step/s, eps=0.27, loss=0.0451]   


KeyboardInterrupt: 

## Merging Chunks + Analysis

Collect all `figure3_chunk_*.csv` files (from every session) into `outputs/tables/` on one machine, then run the cells below.

In [ ]:
def validate_summary_table(df):
    missing = [column for column in FIGURE3_SCHEMA if column not in df.columns]
    if missing:
        raise ValueError(f"Missing required Figure 3 columns: {missing}")


def load_summary_table(path):
    df = pd.read_csv(path)
    validate_summary_table(df)
    return df


import glob

chunk_paths = sorted(glob.glob(str(TABLES_DIR / "figure3_chunk_*.csv")))
if chunk_paths:
    df = pd.concat([load_summary_table(p) for p in chunk_paths], ignore_index=True)
    print(f"merged {len(chunk_paths)} chunks -> {len(df)} rows")
else:
    df = smoke_df  # fall back to the smoke results so the analysis cells below run
    print("no chunk files found; using smoke results")

In [ ]:
def figure3_long_table(df):
    validate_summary_table(df)
    rows = []
    for _, row in df.iterrows():
        for panel in ["weight_norm", "weight_rank", "dead_units", "feature_rank"]:
            rows.append({**row.to_dict(), "panel": panel,
                         "statistic_name": panel, "statistic_value": float(row[panel])})
    return pd.DataFrame(rows)


def correlation_summary(df):
    long_df = figure3_long_table(df)
    rows = []
    for spec in PANEL_SPECS:
        panel_df = long_df[long_df["panel"] == spec.panel]
        for condition in spec.conditions:
            group = panel_df[panel_df[spec.condition_column] == condition]
            x = group["statistic_value"].to_numpy()
            y = group["plasticity_loss"].to_numpy()
            if not _has_regression_support(x, y):
                corr, slope = np.nan, np.nan
            else:
                corr = float(np.corrcoef(x, y)[0, 1])
                slope = float(np.polyfit(x, y, 1)[0])
            rows.append({"panel": spec.panel, "condition": condition, "n": len(group),
                         "pearson_r": corr, "linear_slope": slope})
    return pd.DataFrame(rows)


correlation_summary(df)

In [ ]:
def plot_figure3(df, save_path=None):
    import matplotlib.pyplot as plt

    long_df = figure3_long_table(df)
    fig, axes = plt.subplots(1, 4, figsize=(15, 3.4), constrained_layout=True)
    fig.suptitle("Falsification of explanations of plasticity", fontsize=13, y=1.04)

    for ax, spec in zip(axes, PANEL_SPECS):
        panel_df = long_df[long_df["panel"] == spec.panel]
        for condition in spec.conditions:
            group = panel_df[panel_df[spec.condition_column] == condition]
            if group.empty:
                continue
            color = COLORS[condition]
            ax.scatter(group["statistic_value"], group["plasticity_loss"],
                       s=24, alpha=0.62, color=color, edgecolor="none", label=condition)
            x = group["statistic_value"].to_numpy()
            y = group["plasticity_loss"].to_numpy()
            if _has_regression_support(x, y):
                slope, intercept = np.polyfit(x, y, 1)
                xs = np.linspace(float(x.min()), float(x.max()), 100)
                ax.plot(xs, slope * xs + intercept, color=color, linewidth=1.6)
        ax.set_title(spec.title, fontsize=11)
        ax.set_xlabel(spec.x_label)
        ax.grid(True, linewidth=0.6, alpha=0.55)
        ax.legend(frameon=True, fontsize=9)

    axes[0].set_ylabel("Plasticity loss")

    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=180, bbox_inches="tight")
        print(f"Saved figure to {save_path}")
    return fig


fig = plot_figure3(df, save_path=FIGURES_DIR / "figure3_falsification_reproduction.png")